<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_17_Improve_RAG_Precision_with_Metadata_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📚 Day 17 — Improve RAG Precision with Metadata Filtering

## 🎯 Objective

The goal of Day 17 was to improve the retrieval precision of a FAISS-based RAG pipeline by adding **structured metadata filtering** alongside vector similarity search.

Instead of retrieving documents only based on semantic similarity, each document was enhanced with metadata such as:

- **Source**
- **Category**
- **Date**
- **Document Type**

---

## 🏗️ Architecture

### Before Metadata Filtering

```text
User Query
    ↓
Sentence Transformer
    ↓
Query Embedding
    ↓
FAISS Similarity Search
    ↓
Top-K Documents
    ↓
RAG Context

In [1]:
# ============================================================
# DAY 17 — STEP 1
# CREATE METADATA-ENABLED KNOWLEDGE BASE
# ============================================================

DOCUMENTS = [
    {
        "content": "FAISS is a library for efficient similarity search and clustering of dense vectors.",
        "metadata": {
            "source": "FAISS Technical Documentation",
            "category": "technical",
            "date": "2024-03-15",
            "document_type": "reference"
        }
    },

    {
        "content": "FAISS can be used to build vector indexes for large-scale semantic search applications.",
        "metadata": {
            "source": "FAISS Engineering Guide",
            "category": "technical",
            "date": "2023-08-10",
            "document_type": "reference"
        }
    },

    {
        "content": "Retrieval Augmented Generation combines document retrieval with language model generation.",
        "metadata": {
            "source": "RAG Research Notes",
            "category": "research",
            "date": "2024-06-20",
            "document_type": "research"
        }
    },

    {
        "content": "RAG systems retrieve relevant context before generating an answer to a user query.",
        "metadata": {
            "source": "AI Engineering Handbook",
            "category": "technical",
            "date": "2025-01-12",
            "document_type": "guide"
        }
    },

    {
        "content": "Metadata filtering can restrict vector search to documents belonging to a particular category.",
        "metadata": {
            "source": "Vector Search Guide",
            "category": "technical",
            "date": "2025-02-18",
            "document_type": "guide"
        }
    },

    {
        "content": "Marketing teams can use AI systems to summarize customer feedback and identify trends.",
        "metadata": {
            "source": "AI Marketing Overview",
            "category": "marketing",
            "date": "2022-11-05",
            "document_type": "overview"
        }
    },

    {
        "content": "Modern AI applications increasingly use vector databases for semantic document retrieval.",
        "metadata": {
            "source": "AI Industry Report",
            "category": "business",
            "date": "2023-05-22",
            "document_type": "report"
        }
    },

    {
        "content": "Dense vector embeddings represent text as numerical vectors that capture semantic relationships.",
        "metadata": {
            "source": "Embedding Fundamentals",
            "category": "technical",
            "date": "2024-09-11",
            "document_type": "reference"
        }
    },

    {
        "content": "Semantic search finds documents based on meaning rather than exact keyword matching.",
        "metadata": {
            "source": "Search Systems Guide",
            "category": "technical",
            "date": "2025-03-01",
            "document_type": "guide"
        }
    },

    {
        "content": "Companies are increasingly adopting generative AI to improve productivity and automate repetitive tasks.",
        "metadata": {
            "source": "Enterprise AI Report",
            "category": "business",
            "date": "2021-07-14",
            "document_type": "report"
        }
    }
]


# ------------------------------------------------------------
# CREATE SIMPLE TEXT LIST FOR EMBEDDINGS
# ------------------------------------------------------------

texts = [
    doc["content"]
    for doc in DOCUMENTS
]


# ------------------------------------------------------------
# DISPLAY KNOWLEDGE BASE
# ------------------------------------------------------------

print("=" * 80)
print("METADATA-ENABLED KNOWLEDGE BASE")
print("=" * 80)

for i, doc in enumerate(DOCUMENTS):

    print(f"\nDocument {i}")
    print("-" * 50)

    print("Content:")
    print(doc["content"])

    print("\nMetadata:")
    print(doc["metadata"])


print("\nTotal documents:", len(DOCUMENTS))

METADATA-ENABLED KNOWLEDGE BASE

Document 0
--------------------------------------------------
Content:
FAISS is a library for efficient similarity search and clustering of dense vectors.

Metadata:
{'source': 'FAISS Technical Documentation', 'category': 'technical', 'date': '2024-03-15', 'document_type': 'reference'}

Document 1
--------------------------------------------------
Content:
FAISS can be used to build vector indexes for large-scale semantic search applications.

Metadata:
{'source': 'FAISS Engineering Guide', 'category': 'technical', 'date': '2023-08-10', 'document_type': 'reference'}

Document 2
--------------------------------------------------
Content:
Retrieval Augmented Generation combines document retrieval with language model generation.

Metadata:
{'source': 'RAG Research Notes', 'category': 'research', 'date': '2024-06-20', 'document_type': 'research'}

Document 3
--------------------------------------------------
Content:
RAG systems retrieve relevant context be

In [2]:
# ============================================================
# DAY 17 — STEP 2
# CREATE EMBEDDINGS + FAISS INDEX + METADATA MAPPING
# ============================================================

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 1. LOAD EMBEDDING MODEL
# ------------------------------------------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")


# ------------------------------------------------------------
# 2. EXTRACT DOCUMENT TEXT
# ------------------------------------------------------------

texts = [
    doc["content"]
    for doc in DOCUMENTS
]


# ------------------------------------------------------------
# 3. CREATE EMBEDDINGS
# ------------------------------------------------------------

embeddings = model.encode(
    texts,
    convert_to_numpy=True
).astype("float32")


# Normalize embeddings so Inner Product ≈ Cosine Similarity
faiss.normalize_L2(embeddings)


print("Embedding shape:", embeddings.shape)


# ------------------------------------------------------------
# 4. CREATE FAISS INDEX
# ------------------------------------------------------------

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)


print("FAISS index created successfully!")
print("Number of vectors:", index.ntotal)


# ------------------------------------------------------------
# 5. VERIFY METADATA MAPPING
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VECTOR ↔ DOCUMENT ↔ METADATA MAPPING")
print("=" * 80)

for i, doc in enumerate(DOCUMENTS):

    print(f"\nVector ID: {i}")

    print(
        "Content:",
        doc["content"]
    )

    print(
        "Source:",
        doc["metadata"]["source"]
    )

    print(
        "Category:",
        doc["metadata"]["category"]
    )

    print(
        "Date:",
        doc["metadata"]["date"]
    )

    print(
        "Document Type:",
        doc["metadata"]["document_type"]
    )


# ------------------------------------------------------------
# 6. BASIC RETRIEVAL FUNCTION
# ------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({
            "content": DOCUMENTS[idx]["content"],
            "metadata": DOCUMENTS[idx]["metadata"],
            "similarity_score": round(
                float(score),
                4
            )
        })

    return results


# ------------------------------------------------------------
# 7. TEST RETRIEVAL
# ------------------------------------------------------------

query = "How does vector similarity search work?"

results_test = retrieve(
    query,
    k=3
)


print("\n" + "=" * 80)
print("TEST RETRIEVAL")
print("=" * 80)

print("\nQuery:", query)

for i, result in enumerate(
    results_test,
    start=1
):

    print(f"\nResult {i}")
    print("-" * 50)

    print(
        "Score:",
        result["similarity_score"]
    )

    print(
        "Content:",
        result["content"]
    )

    print(
        "Metadata:",
        result["metadata"]
    )

ModuleNotFoundError: No module named 'faiss'

In [3]:
!pip install -q faiss-cpu sentence-transformers numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.5 MB/s eta 0:00:00


In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print("FAISS version:", faiss.__version__)
print("NumPy version:", np.__version__)
print("All libraries loaded successfully! ✅")

FAISS version: 1.15.0
NumPy version: 2.1.3
All libraries loaded successfully! ✅


In [5]:
# ============================================================
# DAY 17 — STEP 2: CREATE EMBEDDINGS + FAISS INDEX
# ============================================================

# 1. Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded! ✅")


# 2. Extract text from metadata documents
texts = [
    doc["content"]
    for doc in DOCUMENTS
]


# 3. Create embeddings
embeddings = model.encode(
    texts,
    convert_to_numpy=True
).astype("float32")


# 4. Normalize embeddings
# Inner Product after normalization ≈ cosine similarity
faiss.normalize_L2(embeddings)

print("Embeddings created!")
print("Embedding shape:", embeddings.shape)


# 5. Create FAISS index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created! ✅")
print("Total vectors:", index.ntotal)


# 6. Basic retrieval function
def retrieve(query, k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "content": DOCUMENTS[idx]["content"],
            "metadata": DOCUMENTS[idx]["metadata"],
            "similarity_score": round(
                float(score), 4
            )
        })

    return results


# 7. Test retrieval
query = "How does vector similarity search work?"

results = retrieve(query, k=3)


print("\n" + "=" * 70)
print("BASIC RETRIEVAL TEST")
print("=" * 70)

print("Query:", query)

for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("-" * 50)

    print("Score:", result["similarity_score"])
    print("Content:", result["content"])
    print("Metadata:", result["metadata"])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded! ✅
Embeddings created!
Embedding shape: (10, 384)
FAISS index created! ✅
Total vectors: 10

BASIC RETRIEVAL TEST
Query: How does vector similarity search work?

Result 1
--------------------------------------------------
Score: 0.6311
Content: FAISS is a library for efficient similarity search and clustering of dense vectors.
Metadata: {'source': 'FAISS Technical Documentation', 'category': 'technical', 'date': '2024-03-15', 'document_type': 'reference'}

Result 2
--------------------------------------------------
Score: 0.5138
Content: Modern AI applications increasingly use vector databases for semantic document retrieval.
Metadata: {'source': 'AI Industry Report', 'category': 'business', 'date': '2023-05-22', 'document_type': 'report'}

Result 3
--------------------------------------------------
Score: 0.494
Content: FAISS can be used to build vector indexes for large-scale semantic search applications.
Metadata: {'source': 'FAISS Engineering Guide', 'category

In [6]:
# ============================================================
# DAY 17 — STEP 3
# METADATA FILTERED RETRIEVAL
# ============================================================

from datetime import datetime


# ------------------------------------------------------------
# 1. FILTER FUNCTION
# ------------------------------------------------------------

def filtered_retrieve(query, filters=None, k=3):

    if filters is None:
        filters = {}

    # --------------------------------------------------------
    # FIND DOCUMENTS THAT SATISFY METADATA FILTERS
    # --------------------------------------------------------

    allowed_indices = []

    for i, doc in enumerate(DOCUMENTS):

        metadata = doc["metadata"]

        # Category filter
        if "category" in filters:

            if metadata["category"] != filters["category"]:
                continue

        # Document type filter
        if "document_type" in filters:

            if metadata["document_type"] != filters["document_type"]:
                continue

        # Source filter
        if "source" in filters:

            if metadata["source"] != filters["source"]:
                continue

        # Date filter
        if "date_after" in filters:

            document_date = datetime.strptime(
                metadata["date"],
                "%Y-%m-%d"
            )

            cutoff_date = datetime.strptime(
                filters["date_after"],
                "%Y-%m-%d"
            )

            if document_date < cutoff_date:
                continue

        # If document passed all filters
        allowed_indices.append(i)


    # --------------------------------------------------------
    # NO DOCUMENTS MATCHED
    # --------------------------------------------------------

    if len(allowed_indices) == 0:

        return []


    # --------------------------------------------------------
    # CREATE QUERY EMBEDDING
    # --------------------------------------------------------

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)


    # --------------------------------------------------------
    # SEARCH ALL VECTORS
    # --------------------------------------------------------

    scores, indices = index.search(
        query_embedding,
        len(DOCUMENTS)
    )


    # --------------------------------------------------------
    # KEEP ONLY FILTERED DOCUMENTS
    # --------------------------------------------------------

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if int(idx) in allowed_indices:

            results.append({

                "content":
                    DOCUMENTS[idx]["content"],

                "metadata":
                    DOCUMENTS[idx]["metadata"],

                "similarity_score":
                    round(float(score), 4)
            })

        # Stop after k valid results
        if len(results) == k:
            break


    return results


# ============================================================
# TEST 1 — CATEGORY FILTER
# ============================================================

query = "How does vector similarity search work?"

print("=" * 80)
print("CATEGORY FILTER TEST")
print("=" * 80)

print("\nQuery:", query)

print("\nFilter:")
print({
    "category": "technical"
})


results_category = filtered_retrieve(
    query,
    filters={
        "category": "technical"
    },
    k=3
)


for i, result in enumerate(
    results_category,
    1
):

    print(f"\nResult {i}")
    print("-" * 50)

    print(
        "Score:",
        result["similarity_score"]
    )

    print(
        "Content:",
        result["content"]
    )

    print(
        "Category:",
        result["metadata"]["category"]
    )


# ============================================================
# TEST 2 — DATE FILTER
# ============================================================

print("\n\n" + "=" * 80)
print("DATE FILTER TEST")
print("=" * 80)

print("\nQuery:", query)

print("\nFilter:")
print({
    "date_after": "2024-01-01"
})


results_date = filtered_retrieve(
    query,
    filters={
        "date_after": "2024-01-01"
    },
    k=3
)


for i, result in enumerate(
    results_date,
    1
):

    print(f"\nResult {i}")
    print("-" * 50)

    print(
        "Score:",
        result["similarity_score"]
    )

    print(
        "Content:",
        result["content"]
    )

    print(
        "Date:",
        result["metadata"]["date"]
    )


# ============================================================
# TEST 3 — MULTIPLE FILTERS
# ============================================================

print("\n\n" + "=" * 80)
print("MULTIPLE FILTER TEST")
print("=" * 80)

filters = {
    "category": "technical",
    "date_after": "2024-01-01"
}

print("\nFilters:", filters)


results_multiple = filtered_retrieve(
    query,
    filters=filters,
    k=3
)


for i, result in enumerate(
    results_multiple,
    1
):

    print(f"\nResult {i}")
    print("-" * 50)

    print(
        "Score:",
        result["similarity_score"]
    )

    print(
        "Content:",
        result["content"]
    )

    print(
        "Metadata:",
        result["metadata"]
    )


print("\n" + "=" * 80)
print("STEP 3 COMPLETE ✅")
print("=" * 80)

CATEGORY FILTER TEST

Query: How does vector similarity search work?

Filter:
{'category': 'technical'}

Result 1
--------------------------------------------------
Score: 0.6311
Content: FAISS is a library for efficient similarity search and clustering of dense vectors.
Category: technical

Result 2
--------------------------------------------------
Score: 0.494
Content: FAISS can be used to build vector indexes for large-scale semantic search applications.
Category: technical

Result 3
--------------------------------------------------
Score: 0.4387
Content: Semantic search finds documents based on meaning rather than exact keyword matching.
Category: technical


DATE FILTER TEST

Query: How does vector similarity search work?

Filter:
{'date_after': '2024-01-01'}

Result 1
--------------------------------------------------
Score: 0.6311
Content: FAISS is a library for efficient similarity search and clustering of dense vectors.
Date: 2024-03-15

Result 2
----------------------------

In [7]:
# ============================================================
# DAY 17 — STEP 4
# FILTERED VS UNFILTERED RETRIEVAL PRECISION
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. FIVE TEST QUERIES
# ------------------------------------------------------------

test_cases = [

    {
        "query": "How does FAISS perform vector similarity search?",
        "filters": {
            "category": "technical"
        },
        "expected_category": "technical"
    },

    {
        "query": "What is retrieval augmented generation?",
        "filters": {
            "category": "research"
        },
        "expected_category": "research"
    },

    {
        "query": "How do embeddings represent text?",
        "filters": {
            "category": "technical"
        },
        "expected_category": "technical"
    },

    {
        "query": "How can companies use AI to improve productivity?",
        "filters": {
            "category": "business"
        },
        "expected_category": "business"
    },

    {
        "query": "How does semantic search find relevant documents?",
        "filters": {
            "category": "technical",
            "date_after": "2024-01-01"
        },
        "expected_category": "technical"
    }
]


# ------------------------------------------------------------
# 2. PRECISION FUNCTION
# ------------------------------------------------------------

def calculate_precision(
    retrieved,
    expected_category
):

    if len(retrieved) == 0:
        return 0.0

    relevant = 0

    for result in retrieved:

        if (
            result["metadata"]["category"]
            == expected_category
        ):
            relevant += 1

    return relevant / len(retrieved)


# ------------------------------------------------------------
# 3. RUN EXPERIMENT
# ------------------------------------------------------------

experiment_results = []


for i, test in enumerate(
    test_cases,
    start=1
):

    query = test["query"]

    filters = test["filters"]

    expected_category = test[
        "expected_category"
    ]


    # --------------------------------------------------------
    # WITHOUT FILTER
    # --------------------------------------------------------

    unfiltered = retrieve(
        query,
        k=3
    )


    # --------------------------------------------------------
    # WITH FILTER
    # --------------------------------------------------------

    filtered = filtered_retrieve(
        query,
        filters=filters,
        k=3
    )


    # --------------------------------------------------------
    # PRECISION
    # --------------------------------------------------------

    precision_without = calculate_precision(
        unfiltered,
        expected_category
    )

    precision_with = calculate_precision(
        filtered,
        expected_category
    )


    improvement = (
        precision_with
        - precision_without
    )


    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    experiment_results.append({

        "Test ID":
            i,

        "Query":
            query,

        "Filters":
            str(filters),

        "Precision Without Filter":
            round(
                precision_without,
                2
            ),

        "Precision With Filter":
            round(
                precision_with,
                2
            ),

        "Improvement":
            round(
                improvement,
                2
            )
    })


# ------------------------------------------------------------
# 4. CREATE DATAFRAME
# ------------------------------------------------------------

comparison_df = pd.DataFrame(
    experiment_results
)


# ------------------------------------------------------------
# 5. DISPLAY COMPARISON
# ------------------------------------------------------------

print("=" * 100)
print("METADATA FILTERING — PRECISION COMPARISON")
print("=" * 100)

display(
    comparison_df
)


# ------------------------------------------------------------
# 6. AVERAGE PRECISION
# ------------------------------------------------------------

average_without = comparison_df[
    "Precision Without Filter"
].mean()

average_with = comparison_df[
    "Precision With Filter"
].mean()

average_improvement = (
    average_with
    - average_without
)


print("\n" + "=" * 70)
print("OVERALL RESULTS")
print("=" * 70)

print(
    f"Average Precision WITHOUT Filter: "
    f"{average_without:.2f}"
)

print(
    f"Average Precision WITH Filter:    "
    f"{average_with:.2f}"
)

print(
    f"Average Improvement:              "
    f"{average_improvement:+.2f}"
)


# ------------------------------------------------------------
# 7. SHOW ACTUAL RETRIEVED DOCUMENTS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("DETAILED RETRIEVAL COMPARISON")
print("=" * 100)


for i, test in enumerate(
    test_cases,
    start=1
):

    print("\n" + "#" * 100)

    print(
        f"TEST {i}: {test['query']}"
    )

    print(
        "Filters:",
        test["filters"]
    )


    # Unfiltered
    print("\nWITHOUT FILTER:")

    unfiltered = retrieve(
        test["query"],
        k=3
    )

    for j, result in enumerate(
        unfiltered,
        start=1
    ):

        print(
            f"\n  {j}. Score: "
            f"{result['similarity_score']}"
        )

        print(
            "     Category:",
            result["metadata"]["category"]
        )

        print(
            "     Source:",
            result["metadata"]["source"]
        )

        print(
            "     Content:",
            result["content"]
        )


    # Filtered
    print("\nWITH FILTER:")

    filtered = filtered_retrieve(
        test["query"],
        filters=test["filters"],
        k=3
    )

    for j, result in enumerate(
        filtered,
        start=1
    ):

        print(
            f"\n  {j}. Score: "
            f"{result['similarity_score']}"
        )

        print(
            "     Category:",
            result["metadata"]["category"]
        )

        print(
            "     Source:",
            result["metadata"]["source"]
        )

        print(
            "     Content:",
            result["content"]
        )


# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------

comparison_df.to_csv(
    "metadata_precision_comparison.csv",
    index=False
)


print("\n" + "=" * 80)
print("RESULTS SAVED")
print("=" * 80)

print(
    "metadata_precision_comparison.csv"
)

print("\nSTEP 4 COMPLETE ✅")

METADATA FILTERING — PRECISION COMPARISON


,Test ID,Query,Filters,Precision Without Filter,Precision With Filter,Improvement
0,1,How does FAISS perform vector similarity search?,{'category': 'technical'},0.67,1.0,0.33
1,2,What is retrieval augmented generation?,{'category': 'research'},0.33,1.0,0.67
2,3,How do embeddings represent text?,{'category': 'technical'},0.33,1.0,0.67
3,4,How can companies use AI to improve productivity?,{'category': 'business'},0.67,1.0,0.33
4,5,How does semantic search find relevant documents?,"{'category': 'technical', 'date_after': '2024-...",1.00,1.0,0.00



OVERALL RESULTS
Average Precision WITHOUT Filter: 0.60
Average Precision WITH Filter:    1.00
Average Improvement:              +0.40

DETAILED RETRIEVAL COMPARISON

####################################################################################################
TEST 1: How does FAISS perform vector similarity search?
Filters: {'category': 'technical'}

WITHOUT FILTER:

  1. Score: 0.7085
     Category: technical
     Source: FAISS Technical Documentation
     Content: FAISS is a library for efficient similarity search and clustering of dense vectors.

  2. Score: 0.6391
     Category: technical
     Source: FAISS Engineering Guide
     Content: FAISS can be used to build vector indexes for large-scale semantic search applications.

  3. Score: 0.4323
     Category: business
     Source: AI Industry Report
     Content: Modern AI applications increasingly use vector databases for semantic document retrieval.

WITH FILTER:

  1. Score: 0.7085
     Category: technical
     Source: F

In [8]:
# ============================================================
# DAY 17 — STEP 5
# DATE FILTER + EDGE CASES
# ============================================================

from datetime import datetime


# ------------------------------------------------------------
# 1. ROBUST METADATA FILTER FUNCTION
# ------------------------------------------------------------

def filtered_retrieve_v2(query, filters=None, k=3):

    if filters is None:
        filters = {}

    allowed_indices = []

    for i, doc in enumerate(DOCUMENTS):

        metadata = doc.get("metadata", {})

        # ----------------------------------------------------
        # CATEGORY FILTER
        # ----------------------------------------------------

        if "category" in filters:

            category = metadata.get("category")

            if category is None:
                continue

            if category != filters["category"]:
                continue


        # ----------------------------------------------------
        # DOCUMENT TYPE FILTER
        # ----------------------------------------------------

        if "document_type" in filters:

            document_type = metadata.get(
                "document_type"
            )

            if document_type is None:
                continue

            if document_type != filters["document_type"]:
                continue


        # ----------------------------------------------------
        # SOURCE FILTER
        # ----------------------------------------------------

        if "source" in filters:

            source = metadata.get("source")

            if source is None:
                continue

            if source != filters["source"]:
                continue


        # ----------------------------------------------------
        # DATE CUTOFF FILTER
        # ----------------------------------------------------

        if "date_after" in filters:

            document_date_string = metadata.get(
                "date"
            )

            # Missing date → cannot satisfy date filter
            if document_date_string is None:
                continue

            try:

                document_date = datetime.strptime(
                    document_date_string,
                    "%Y-%m-%d"
                )

                cutoff_date = datetime.strptime(
                    filters["date_after"],
                    "%Y-%m-%d"
                )

            except ValueError:

                # Invalid date → exclude document
                continue

            if document_date < cutoff_date:
                continue


        # ----------------------------------------------------
        # DOCUMENT PASSED ALL FILTERS
        # ----------------------------------------------------

        allowed_indices.append(i)


    # --------------------------------------------------------
    # NO MATCHING DOCUMENTS
    # --------------------------------------------------------

    if not allowed_indices:
        return []


    # --------------------------------------------------------
    # QUERY EMBEDDING
    # --------------------------------------------------------

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)


    # --------------------------------------------------------
    # FAISS SEARCH
    # --------------------------------------------------------

    scores, indices = index.search(
        query_embedding,
        len(DOCUMENTS)
    )


    # --------------------------------------------------------
    # APPLY METADATA FILTER TO SEARCH RESULTS
    # --------------------------------------------------------

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        idx = int(idx)

        if idx in allowed_indices:

            results.append({

                "content":
                    DOCUMENTS[idx]["content"],

                "metadata":
                    DOCUMENTS[idx]["metadata"],

                "similarity_score":
                    round(
                        float(score),
                        4
                    )
            })

        if len(results) == k:
            break


    return results


# ============================================================
# TEST 1 — DATE CUTOFF
# ============================================================

print("=" * 80)
print("TEST 1 — DATE CUTOFF")
print("=" * 80)

query = "How do embeddings represent text?"

filters = {
    "date_after": "2024-01-01"
}

results = filtered_retrieve_v2(
    query,
    filters,
    k=5
)

print("\nQuery:", query)
print("Filter:", filters)

for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("-" * 50)

    print(
        "Score:",
        result["similarity_score"]
    )

    print(
        "Date:",
        result["metadata"].get("date")
    )

    print(
        "Content:",
        result["content"]
    )


# ============================================================
# TEST 2 — MISSING METADATA
# ============================================================

print("\n\n" + "=" * 80)
print("TEST 2 — MISSING METADATA")
print("=" * 80)

# Temporarily add a document with missing metadata

missing_metadata_doc = {
    "content": "This document has incomplete metadata.",
    "metadata": {
        "source": "Unknown Source",
        "category": "technical"
        # date intentionally missing
    }
}

DOCUMENTS.append(
    missing_metadata_doc
)

print("\nAdded document with missing date metadata.")

missing_date_results = filtered_retrieve_v2(
    "technical document information",
    {
        "date_after": "2024-01-01"
    },
    k=5
)

print(
    "\nDocuments returned:",
    len(missing_date_results)
)

print(
    "Missing-date document excluded safely ✅"
)


# ============================================================
# REMOVE TEMPORARY DOCUMENT
# ============================================================

DOCUMENTS.pop()


# ============================================================
# TEST 3 — CONFLICTING CATEGORY
# ============================================================

print("\n\n" + "=" * 80)
print("TEST 3 — CONFLICTING CATEGORY")
print("=" * 80)

conflicting_doc = {
    "content": "This document discusses technical AI products for marketing.",
    "metadata": {
        "source": "Mixed AI Report",
        "category": "marketing",
        "date": "2025-01-01",
        "document_type": "technical"
    }
}

DOCUMENTS.append(
    conflicting_doc
)

print("\nAdded document with:")
print("category = marketing")
print("document_type = technical")


# Category filter
category_results = filtered_retrieve_v2(
    "technical AI information",
    {
        "category": "technical"
    },
    k=5
)

print("\nCategory = technical")

for result in category_results:

    print(
        "\nCategory:",
        result["metadata"].get("category")
    )

    print(
        "Type:",
        result["metadata"].get(
            "document_type"
        )
    )


# Document type filter
type_results = filtered_retrieve_v2(
    "technical AI information",
    {
        "document_type": "technical"
    },
    k=5
)

print("\nDocument Type = technical")

for result in type_results:

    print(
        "\nCategory:",
        result["metadata"].get("category")
    )

    print(
        "Type:",
        result["metadata"].get(
            "document_type"
        )
    )


# Remove temporary document
DOCUMENTS.pop()


# ============================================================
# TEST 4 — MULTIPLE FILTERS
# ============================================================

print("\n\n" + "=" * 80)
print("TEST 4 — CATEGORY + DATE")
print("=" * 80)

filters = {
    "category": "technical",
    "date_after": "2024-01-01"
}

results = filtered_retrieve_v2(
    "semantic search and embeddings",
    filters,
    k=5
)

print("\nFilters:", filters)

for i, result in enumerate(
    results,
    1
):

    print(f"\nResult {i}")

    print(
        "Score:",
        result["similarity_score"]
    )

    print(
        "Category:",
        result["metadata"].get(
            "category"
        )
    )

    print(
        "Date:",
        result["metadata"].get(
            "date"
        )
    )


print("\n" + "=" * 80)
print("STEP 5 COMPLETE ✅")
print("=" * 80)

TEST 1 — DATE CUTOFF

Query: How do embeddings represent text?
Filter: {'date_after': '2024-01-01'}

Result 1
--------------------------------------------------
Score: 0.6987
Date: 2024-09-11
Content: Dense vector embeddings represent text as numerical vectors that capture semantic relationships.

Result 2
--------------------------------------------------
Score: 0.2675
Date: 2024-06-20
Content: Retrieval Augmented Generation combines document retrieval with language model generation.

Result 3
--------------------------------------------------
Score: 0.2254
Date: 2024-03-15
Content: FAISS is a library for efficient similarity search and clustering of dense vectors.

Result 4
--------------------------------------------------
Score: 0.1654
Date: 2025-02-18
Content: Metadata filtering can restrict vector search to documents belonging to a particular category.

Result 5
--------------------------------------------------
Score: 0.1543
Date: 2025-01-12
Content: RAG systems retrieve relevan

In [12]:
# ============================================================
# DAY 17 — STEP 6
# FINAL README - SIMPLE VERSION
# ============================================================

import os

# Get precision values if Step 4 was already executed
if "comparison_df" in globals():

    avg_without = comparison_df[
        "Precision Without Filter"
    ].mean()

    avg_with = comparison_df[
        "Precision With Filter"
    ].mean()

    improvement = avg_with - avg_without

else:

    avg_without = 0
    avg_with = 0
    improvement = 0


# ------------------------------------------------------------
# README LINES
# ------------------------------------------------------------

lines = []

lines.append("# RAG Metadata Filtering - Day 17")
lines.append("")
lines.append("## Overview")
lines.append("")
lines.append(
    "This project improves a FAISS-based RAG pipeline "
    "using metadata filtering."
)

lines.append("")
lines.append("## Objectives")
lines.append("")
lines.append("- Add source metadata")
lines.append("- Add category metadata")
lines.append("- Add date metadata")
lines.append("- Add document type metadata")
lines.append("- Implement filtered_retrieve()")
lines.append("- Implement category filtering")
lines.append("- Implement date filtering")
lines.append("- Compare filtered and unfiltered retrieval")
lines.append("- Measure retrieval precision")
lines.append("- Document edge cases and limitations")

lines.append("")
lines.append("## Technology Stack")
lines.append("")
lines.append("- Python")
lines.append("- FAISS")
lines.append("- Sentence Transformers")
lines.append("- NumPy")
lines.append("- Pandas")
lines.append("- Embedding Model: all-MiniLM-L6-v2")
lines.append("- OpenAI API: Not required")

lines.append("")
lines.append("## Metadata Structure")
lines.append("")
lines.append("Each document contains content and metadata:")
lines.append("")
lines.append("source")
lines.append("category")
lines.append("date")
lines.append("document_type")

lines.append("")
lines.append("## Architecture Before Filtering")
lines.append("")
lines.append("User Query")
lines.append("    |")
lines.append("    v")
lines.append("Query Embedding")
lines.append("    |")
lines.append("    v")
lines.append("FAISS Similarity Search")
lines.append("    |")
lines.append("    v")
lines.append("Top-K Documents")
lines.append("    |")
lines.append("    v")
lines.append("RAG Context")

lines.append("")
lines.append("## Architecture After Filtering")
lines.append("")
lines.append("User Query")
lines.append("    |")
lines.append("    v")
lines.append("Metadata Filters")
lines.append("    |")
lines.append("    +-- Category")
lines.append("    +-- Date")
lines.append("    +-- Document Type")
lines.append("    +-- Source")
lines.append("    |")
lines.append("    v")
lines.append("Allowed Documents")
lines.append("    |")
lines.append("    v")
lines.append("FAISS Similarity Search")
lines.append("    |")
lines.append("    v")
lines.append("Top-K Results")
lines.append("    |")
lines.append("    v")
lines.append("RAG Context")

lines.append("")
lines.append("## Filtered Retrieval")
lines.append("")
lines.append(
    "The main function is filtered_retrieve(query, filters, k)."
)

lines.append("")
lines.append("Example category filter:")
lines.append("")
lines.append(
    'filtered_retrieve(query, {"category": "technical"}, k=3)'
)

lines.append("")
lines.append("Example date filter:")
lines.append("")
lines.append(
    'filtered_retrieve(query, {"date_after": "2024-01-01"}, k=3)'
)

lines.append("")
lines.append("## Precision Experiment")
lines.append("")
lines.append(
    "Five queries were tested with and without metadata filters."
)

lines.append("")
lines.append(
    "Precision = Relevant Retrieved Documents / "
    "Total Retrieved Documents"
)

lines.append("")
lines.append("## Results")
lines.append("")

lines.append(
    "Average Precision WITHOUT Filter: "
    + str(round(avg_without, 2))
)

lines.append(
    "Average Precision WITH Filter: "
    + str(round(avg_with, 2))
)

lines.append(
    "Average Improvement: "
    + str(round(improvement, 2))
)

lines.append("")
lines.append("The detailed results are stored in:")
lines.append("")
lines.append("metadata_precision_comparison.csv")

lines.append("")
lines.append("## Why Metadata Filtering Helps")
lines.append("")
lines.append(
    "Vector similarity finds semantically similar documents, "
    "but metadata filtering adds structured constraints."
)

lines.append("")
lines.append(
    "For example, a technical query can be restricted to "
    "technical documents using a category filter."
)

lines.append("")
lines.append("## Date Filtering")
lines.append("")
lines.append(
    "The date_after filter removes documents older than "
    "the configured cutoff date."
)

lines.append("")
lines.append("Example:")
lines.append("")
lines.append(
    'date_after = "2024-01-01"'
)

lines.append("")
lines.append("## Edge Case 1 - Missing Metadata")
lines.append("")
lines.append(
    "Documents with missing metadata may be excluded when "
    "that metadata field is required by a filter."
)

lines.append("")
lines.append(
    "This prevents retrieval errors but may exclude a "
    "potentially useful document."
)

lines.append("")
lines.append("## Edge Case 2 - Conflicting Metadata")
lines.append("")
lines.append(
    "A document may have conflicting metadata such as "
    "category=marketing and document_type=technical."
)

lines.append("")
lines.append(
    "Different filters may therefore produce different "
    "results for the same document."
)

lines.append("")
lines.append("## Important Design Decision")
lines.append("")
lines.append(
    "FAISS stores vectors and similarity scores but does "
    "not understand the document metadata."
)

lines.append("")
lines.append(
    "The application maintains the mapping between FAISS "
    "vector IDs and document metadata."
)

lines.append("")
lines.append("## Day 16 vs Day 17")
lines.append("")
lines.append("Day 16:")
lines.append("")
lines.append(
    "Query -> Embedding -> FAISS -> Top-K"
)

lines.append("")
lines.append("Day 17:")
lines.append("")
lines.append(
    "Query -> Metadata Filter -> FAISS -> Top-K"
)

lines.append("")
lines.append("## Benefits")
lines.append("")
lines.append("- Higher retrieval precision")
lines.append("- Category-aware retrieval")
lines.append("- Date-aware retrieval")
lines.append("- Document-type-aware retrieval")
lines.append("- Source-aware retrieval")
lines.append("- Reduced irrelevant context")
lines.append("- Better retrieval control")

lines.append("")
lines.append("## Limitations")
lines.append("")
lines.append("- Missing metadata")
lines.append("- Incorrect metadata")
lines.append("- Conflicting metadata")
lines.append("- Poor chunking")
lines.append("- Poor embeddings")
lines.append("- Ambiguous queries")
lines.append("- Information spread across documents")

lines.append("")
lines.append("## Project Files")
lines.append("")
lines.append("rag_diagnostics.ipynb")
lines.append("results.json")
lines.append("rag_failure_scorecard.csv")
lines.append("metadata_precision_comparison.csv")
lines.append("README.md")
lines.append("requirements.txt")

lines.append("")
lines.append("## Installation")
lines.append("")
lines.append(
    "pip install faiss-cpu sentence-transformers numpy pandas"
)

lines.append("")
lines.append("## Key Learning")
lines.append("")
lines.append(
    "Vector similarity tells us what is semantically similar, "
    "while metadata tells us what is allowed."
)

lines.append("")
lines.append("## Conclusion")
lines.append("")
lines.append(
    "The RAG pipeline was upgraded from pure vector similarity "
    "retrieval to metadata-aware retrieval."
)

lines.append("")
lines.append(
    "Metadata filtering provides an additional control layer "
    "that can improve retrieval precision and make RAG systems "
    "more controllable."
)


# ------------------------------------------------------------
# CREATE README
# ------------------------------------------------------------

readme = "\n".join(lines)

with open(
    "README.md",
    "w",
    encoding="utf-8"
) as file:

    file.write(readme)


# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("=" * 70)
print("DAY 17 README CREATED SUCCESSFULLY")
print("=" * 70)

print("README exists:", os.path.exists("README.md"))

print(
    "Average precision without filter:",
    round(avg_without, 2)
)

print(
    "Average precision with filter:",
    round(avg_with, 2)
)

print(
    "Improvement:",
    round(improvement, 2)
)

print("\nFile created:")
print("README.md")

print("\nDAY 17 COMPLETE!")

DAY 17 README CREATED SUCCESSFULLY
README exists: True
Average precision without filter: 0.6
Average precision with filter: 1.0
Improvement: 0.4

File created:
README.md

DAY 17 COMPLETE!


In [13]:
# ============================================================
# DAY 17 — STEP 7
# FINAL PROJECT VERIFICATION
# ============================================================

import os

print("=" * 80)
print("DAY 17 — FINAL SUBMISSION CHECK")
print("=" * 80)


# ------------------------------------------------------------
# CHECK 1 — DOCUMENTS
# ------------------------------------------------------------

if "DOCUMENTS" in globals():
    print("✅ Metadata knowledge base created")
    print("   Documents:", len(DOCUMENTS))
else:
    print("❌ DOCUMENTS not found")


# ------------------------------------------------------------
# CHECK 2 — EMBEDDING MODEL
# ------------------------------------------------------------

if "model" in globals():
    print("✅ Sentence Transformer loaded")
else:
    print("❌ Embedding model not found")


# ------------------------------------------------------------
# CHECK 3 — FAISS
# ------------------------------------------------------------

if "index" in globals():
    print("✅ FAISS index created")
    print("   Vectors:", index.ntotal)
else:
    print("❌ FAISS index not found")


# ------------------------------------------------------------
# CHECK 4 — RETRIEVAL
# ------------------------------------------------------------

if "retrieve" in globals():
    print("✅ Basic retrieve() implemented")
else:
    print("❌ retrieve() missing")


# ------------------------------------------------------------
# CHECK 5 — FILTERED RETRIEVAL
# ------------------------------------------------------------

if "filtered_retrieve" in globals():
    print("✅ filtered_retrieve() implemented")
else:
    print("❌ filtered_retrieve() missing")


# ------------------------------------------------------------
# CHECK 6 — IMPROVED FILTERED RETRIEVAL
# ------------------------------------------------------------

if "filtered_retrieve_v2" in globals():
    print("✅ Date + edge-case filtering implemented")
else:
    print("❌ filtered_retrieve_v2() missing")


# ------------------------------------------------------------
# CHECK 7 — PRECISION EXPERIMENT
# ------------------------------------------------------------

if os.path.exists(
    "metadata_precision_comparison.csv"
):

    print(
        "✅ Five-query precision comparison created"
    )

else:

    print(
        "❌ Precision comparison CSV missing"
    )


# ------------------------------------------------------------
# CHECK 8 — README
# ------------------------------------------------------------

if os.path.exists("README.md"):

    print("✅ README.md created")

else:

    print("❌ README.md missing")


# ------------------------------------------------------------
# CHECK 9 — DAY 16 FILES
# ------------------------------------------------------------

if os.path.exists("results.json"):

    print("✅ Day 16 results.json found")

else:

    print("⚠️ results.json not found")


if os.path.exists(
    "rag_failure_scorecard.csv"
):

    print(
        "✅ Day 16 scorecard found"
    )

else:

    print(
        "⚠️ rag_failure_scorecard.csv not found"
    )


# ------------------------------------------------------------
# FINAL FILE LIST
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PROJECT FILES")
print("=" * 80)

files = [
    "README.md",
    "metadata_precision_comparison.csv",
    "results.json",
    "rag_failure_scorecard.csv",
    "rag_diagnostics.ipynb",
    "requirements.txt"
]

for filename in files:

    if os.path.exists(filename):
        print("✅", filename)
    else:
        print("❌", filename)


# ------------------------------------------------------------
# FINAL MESSAGE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DAY 17 VERIFICATION COMPLETE 🚀")
print("=" * 80)

print("""
Your Day 17 project now contains:

1. Metadata-enabled documents
2. FAISS vector search
3. Category filtering
4. Date filtering
5. Document-type filtering
6. Source filtering
7. filtered_retrieve()
8. Five-query comparison
9. Precision measurement
10. Missing metadata handling
11. Conflicting metadata analysis
12. Updated architecture documentation
13. Final README
""")

DAY 17 — FINAL SUBMISSION CHECK
✅ Metadata knowledge base created
   Documents: 10
✅ Sentence Transformer loaded
✅ FAISS index created
   Vectors: 10
✅ Basic retrieve() implemented
✅ filtered_retrieve() implemented
✅ Date + edge-case filtering implemented
✅ Five-query precision comparison created
✅ README.md created
⚠️ results.json not found
⚠️ rag_failure_scorecard.csv not found

PROJECT FILES
✅ README.md
✅ metadata_precision_comparison.csv
❌ results.json
❌ rag_failure_scorecard.csv
❌ rag_diagnostics.ipynb
❌ requirements.txt

DAY 17 VERIFICATION COMPLETE 🚀

Your Day 17 project now contains:

1. Metadata-enabled documents
2. FAISS vector search
3. Category filtering
4. Date filtering
5. Document-type filtering
6. Source filtering
7. filtered_retrieve()
8. Five-query comparison
9. Precision measurement
10. Missing metadata handling
11. Conflicting metadata analysis
12. Updated architecture documentation
13. Final README

